## **Gold Layer - Creating data model using fact and dimension tables**

### **Importing libraries**

In [82]:
import pandas as pd
import numpy as np

### ***Read Silver layer***

In [83]:
Gold_df = pd.read_csv(
    "/Users/folder/project_real_estate/data/raw/Silver.csv"
)

### **Configure pandas display settings**

In [84]:
pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.max_rows = None
pd.options.display.max_columns = None

### delete unuseful column and rows

In [85]:
Gold_df = Gold_df.drop(columns=["url"])
Gold_df = Gold_df.drop(columns=["description"])
Gold_df = Gold_df.drop(columns=["size_sqft"])

In [86]:
# Remove impossible property sizes
Gold_df = Gold_df[
    (Gold_df["size_sqm"] >= 20) &
    (Gold_df["size_sqm"] <= 1000)
]

# Recalculate price per square meter
Gold_df["price_per_sqm"] = Gold_df["price"] / Gold_df["size_sqm"]

# Remove extreme price_per_sqm outliers (1st–99th percentile)
low = Gold_df["price_per_sqm"].quantile(0.01)
high = Gold_df["price_per_sqm"].quantile(0.99)

Gold_df = Gold_df[
    (Gold_df["price_per_sqm"] >= low) &
    (Gold_df["price_per_sqm"] <= high)
]

# Reset index
Gold_df.reset_index(drop=True, inplace=True)

# Display summary
print(f"Gold layer shape: {Gold_df.shape}")
Gold_df.head()

Gold layer shape: (18871, 15)


,price,type,available_from,payment_method,down_payment,compound,district,city,governorate,size_sqm,maid_room,bedrooms,bathrooms_plus,bathrooms,price_per_sqm
0,"8,000,000.00",Chalet,2025-08-31,Cash,"1,200,000.00",Swan Lake Gouna,Al Gouna,Hurghada,Red Sea,68,True,1.00,False,1.00,"117,647.06"
1,"25,000,000.00",Villa,2025-09-02,Cash,"2,100,000.00",Karmell,New Zayed City,Sheikh Zayed City,Giza,220,False,4.00,False,4.00,"113,636.36"
2,"15,135,000.00",Chalet,2025-08-19,Cash,"1,513,000.00",Azha North,NaN,Ras Al Hekma,North Coast,118,False,2.00,False,2.00,"128,262.71"
3,"12,652,000.00",Apartment,2025-08-26,Installments,"1,260,000.00",Taj City,5th Settlement Compounds,"The 5th Settlement, New Cairo City",Cairo,166,False,3.00,False,2.00,"76,216.87"
4,"45,250,000.00",Villa,2025-09-02,Cash,"2,262,500.00",Granville,NaN,New Capital City,Cairo,400,False,7.00,False,7.00,"113,125.00"


In [87]:
Gold_df.head()

,price,type,available_from,payment_method,down_payment,compound,district,city,governorate,size_sqm,maid_room,bedrooms,bathrooms_plus,bathrooms,price_per_sqm
0,"8,000,000.00",Chalet,2025-08-31,Cash,"1,200,000.00",Swan Lake Gouna,Al Gouna,Hurghada,Red Sea,68,True,1.00,False,1.00,"117,647.06"
1,"25,000,000.00",Villa,2025-09-02,Cash,"2,100,000.00",Karmell,New Zayed City,Sheikh Zayed City,Giza,220,False,4.00,False,4.00,"113,636.36"
2,"15,135,000.00",Chalet,2025-08-19,Cash,"1,513,000.00",Azha North,NaN,Ras Al Hekma,North Coast,118,False,2.00,False,2.00,"128,262.71"
3,"12,652,000.00",Apartment,2025-08-26,Installments,"1,260,000.00",Taj City,5th Settlement Compounds,"The 5th Settlement, New Cairo City",Cairo,166,False,3.00,False,2.00,"76,216.87"
4,"45,250,000.00",Villa,2025-09-02,Cash,"2,262,500.00",Granville,NaN,New Capital City,Cairo,400,False,7.00,False,7.00,"113,125.00"


### add useful column

In [88]:
Gold_df["price_per_sqm"] = (
    Gold_df["price"] / Gold_df["size_sqm"]
).round(2)

In [89]:
Gold_df["available_from"] = pd.to_datetime(
    Gold_df["available_from"],
    format="%Y-%m-%d",
    errors="coerce"
)

In [90]:
print(Gold_df["available_from"].dtype)

datetime64[ns]


In [91]:
Gold_df["available_year"] = Gold_df["available_from"].dt.year.astype("Int64")
Gold_df["available_month"] = Gold_df["available_from"].dt.month.astype("Int64")
Gold_df["available_month_name"] = Gold_df["available_from"].dt.month_name()
Gold_df["available_quarter"] = Gold_df["available_from"].dt.quarter.astype("Int64")
Gold_df["available_day"] = Gold_df["available_from"].dt.day_name()

In [92]:
mask = Gold_df["down_payment"].between(1, 100)

# Start with the original values
Gold_df["down_payment_amount"] = Gold_df["down_payment"]

# Replace percentage values with their EGP equivalent
Gold_df.loc[mask, "down_payment_amount"] = (
    Gold_df.loc[mask, "price"] *
    Gold_df.loc[mask, "down_payment"] / 100
).round()

Gold_df["down_payment_amount"] = Gold_df["down_payment_amount"].astype("Int64")

Gold_df["down_payment_percentage"] = (
    Gold_df["down_payment_amount"] / Gold_df["price"]
).round(4)

In [93]:
Gold_df.head()

,price,type,available_from,payment_method,down_payment,compound,district,city,governorate,size_sqm,maid_room,bedrooms,bathrooms_plus,bathrooms,price_per_sqm,available_year,available_month,available_month_name,available_quarter,available_day,down_payment_amount,down_payment_percentage
0,"8,000,000.00",Chalet,2025-08-31,Cash,"1,200,000.00",Swan Lake Gouna,Al Gouna,Hurghada,Red Sea,68,True,1.00,False,1.00,"117,647.06",2025,8,August,3,Sunday,1200000,0.15
1,"25,000,000.00",Villa,2025-09-02,Cash,"2,100,000.00",Karmell,New Zayed City,Sheikh Zayed City,Giza,220,False,4.00,False,4.00,"113,636.36",2025,9,September,3,Tuesday,2100000,0.08
2,"15,135,000.00",Chalet,2025-08-19,Cash,"1,513,000.00",Azha North,NaN,Ras Al Hekma,North Coast,118,False,2.00,False,2.00,"128,262.71",2025,8,August,3,Tuesday,1513000,0.10
3,"12,652,000.00",Apartment,2025-08-26,Installments,"1,260,000.00",Taj City,5th Settlement Compounds,"The 5th Settlement, New Cairo City",Cairo,166,False,3.00,False,2.00,"76,216.87",2025,8,August,3,Tuesday,1260000,0.10
4,"45,250,000.00",Villa,2025-09-02,Cash,"2,262,500.00",Granville,NaN,New Capital City,Cairo,400,False,7.00,False,7.00,"113,125.00",2025,9,September,3,Tuesday,2262500,0.05


In [94]:
Gold_df["down_payment_amount"].describe()

count        5,345.00
mean     1,953,003.18
std      3,158,255.11
min            116.00
25%        514,000.00
50%        972,100.00
75%      2,070,000.00
max     68,000,000.00
Name: down_payment_amount, dtype: Float64

In [95]:
Gold_df = Gold_df.drop(columns=["down_payment"])
Gold_df = Gold_df.rename(columns={
    "down_payment_amount": "down_payment"
})

In [96]:
Gold_df["down_payment"].isna().sum()

np.int64(13526)

In [97]:
Gold_df["has_down_payment"] = (
    Gold_df["down_payment"].notna()
)

In [98]:
bins = [0, 2_000_000, 5_000_000, 10_000_000, float("inf")]

labels = [
    "Budget",
    "Mid-range",
    "Premium",
    "Luxury"
]

Gold_df["price_category"] = pd.cut(
    Gold_df["price"],
    bins=bins,
    labels=labels
)

In [99]:
bins = [0, 80, 150, 250, float("inf")]

labels = [
    "Small",
    "Medium",
    "Large",
    "Luxury"
]

Gold_df["size_category"] = pd.cut(
    Gold_df["size_sqm"],
    bins=bins,
    labels=labels
)

In [100]:
Gold_df["bedroom_category"] = pd.cut(
    Gold_df["bedrooms"],
    bins=[0, 2, 4, 6, float("inf")],
    labels=["Small", "Family", "Large", "Luxury"]
)

In [101]:
Gold_df["days_until_available"] = (
    Gold_df["available_from"] - pd.Timestamp.today().normalize()
).dt.days

In [102]:
today = pd.Timestamp.today().normalize()

conditions = [
    Gold_df["available_from"].isna(),
    Gold_df["available_from"] <= today,
    Gold_df["available_from"] <= today + pd.Timedelta(days=90)
]

choices = [
    "Unknown",
    "Available",
    "Coming Soon"
]

Gold_df["availability_status"] = np.select(
    conditions,
    choices,
    default="Future"
)

In [103]:
today = pd.Timestamp.today().normalize()

Gold_df["days_until_available"] = (
    Gold_df["available_from"] - today
).dt.days.clip(lower=0)

In [104]:
Gold_df["price_per_sqm"].describe()

count    18,871.00
mean     73,059.90
std      44,390.71
min       8,717.95
25%      39,699.67
50%      63,622.06
75%      95,652.17
max     266,666.67
Name: price_per_sqm, dtype: float64

In [105]:
bins = [
    0,
    25000,
    50000,
    75000,
    100000,
    150000,
    float("inf")
]

labels = [
    "Very Affordable",
    "Affordable",
    "Mid-range",
    "Premium",
    "Luxury",
    "Ultra Luxury"
]

Gold_df["price_per_sqm_category"] = pd.cut(
    Gold_df["price_per_sqm"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [106]:
Gold_df.nsmallest(
    10,
    "price_per_sqm"
)[["price", "size_sqm", "price_per_sqm", "compound", "city"]]

,price,size_sqm,price_per_sqm,compound,city
5772,"2,720,000.00",312,"8,717.95",Azha,Al Ain Al Sokhna
18221,"5,660,000.00",649,"8,721.11",Upville,6 October City
2779,"1,000,000.00",114,"8,771.93",Cali Coast,Ras Al Hekma
11672,"2,500,000.00",285,"8,771.93",Mountain View iCity October,6 October City
18592,"2,500,000.00",285,"8,771.93",Mountain View iCity October,6 October City
1935,"1,200,000.00",136,"8,823.53",MarVille New Zayed,Sheikh Zayed City
9140,"1,500,000.00",170,"8,823.53",Al Khamayel city,Sheikh Zayed City
10107,"1,500,000.00",170,"8,823.53",Palm Hills October,6 October City
12726,"1,500,000.00",170,"8,823.53",Keeva,6 October City
13944,"1,500,000.00",170,"8,823.53",The Crown,6 October City


In [108]:
Gold_df.insert(0, "property_id", range(1, len(Gold_df) + 1))

In [109]:
Gold_df["price_per_bedroom"] = (
    Gold_df["price"] / Gold_df["bedrooms"]
)

In [110]:
Gold_df["sqm_per_bedroom"] = (
    Gold_df["size_sqm"] / Gold_df["bedrooms"]
)

In [111]:
Gold_df.head()

,property_id,price,type,available_from,payment_method,compound,district,city,governorate,size_sqm,maid_room,bedrooms,bathrooms_plus,bathrooms,price_per_sqm,available_year,available_month,available_month_name,available_quarter,available_day,down_payment,down_payment_percentage,has_down_payment,price_category,size_category,bedroom_category,days_until_available,availability_status,price_per_sqm_category,price_per_bedroom,sqm_per_bedroom
0,1,"8,000,000.00",Chalet,2025-08-31,Cash,Swan Lake Gouna,Al Gouna,Hurghada,Red Sea,68,True,1.00,False,1.00,"117,647.06",2025,8,August,3,Sunday,1200000,0.15,True,Premium,Small,Small,0.00,Available,Luxury,"8,000,000.00",68.00
1,2,"25,000,000.00",Villa,2025-09-02,Cash,Karmell,New Zayed City,Sheikh Zayed City,Giza,220,False,4.00,False,4.00,"113,636.36",2025,9,September,3,Tuesday,2100000,0.08,True,Luxury,Large,Family,0.00,Available,Luxury,"6,250,000.00",55.00
2,3,"15,135,000.00",Chalet,2025-08-19,Cash,Azha North,NaN,Ras Al Hekma,North Coast,118,False,2.00,False,2.00,"128,262.71",2025,8,August,3,Tuesday,1513000,0.10,True,Luxury,Medium,Small,0.00,Available,Luxury,"7,567,500.00",59.00
3,4,"12,652,000.00",Apartment,2025-08-26,Installments,Taj City,5th Settlement Compounds,"The 5th Settlement, New Cairo City",Cairo,166,False,3.00,False,2.00,"76,216.87",2025,8,August,3,Tuesday,1260000,0.10,True,Luxury,Large,Family,0.00,Available,Premium,"4,217,333.33",55.33
4,5,"45,250,000.00",Villa,2025-09-02,Cash,Granville,NaN,New Capital City,Cairo,400,False,7.00,False,7.00,"113,125.00",2025,9,September,3,Tuesday,2262500,0.05,True,Luxury,Luxury,Luxury,0.00,Available,Luxury,"6,464,285.71",57.14


### ***Dimesnion tables***

1.Dim_date

In [112]:
Dim_Date = (
    Gold_df[
        [
            "available_from",
            "available_year",
            "available_month",
            "available_month_name",
            "available_quarter",
            "available_day"
        ]
    ]
    .drop_duplicates()
    .sort_values("available_from")
    .reset_index(drop=True)
)

# Create surrogate key
Dim_Date.insert(0, "date_key", range(1, len(Dim_Date) + 1))

Dim_Date.head()

,date_key,available_from,available_year,available_month,available_month_name,available_quarter,available_day
0,1,2023-06-14,2023,6,June,2,Wednesday
1,2,2023-07-18,2023,7,July,3,Tuesday
2,3,2023-08-03,2023,8,August,3,Thursday
3,4,2023-09-20,2023,9,September,3,Wednesday
4,5,2023-12-24,2023,12,December,4,Sunday


2.Dim_Location

In [114]:
# Create Dim_Location
Dim_Location = (
    Gold_df[
        [
            "compound",
            "district",
            "city",
            "governorate"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Create surrogate key
Dim_Location.insert(0, "location_key", range(1, len(Dim_Location) + 1))

Dim_Location.head(20)

,location_key,compound,district,city,governorate
0,1,Swan Lake Gouna,Al Gouna,Hurghada,Red Sea
1,2,Karmell,New Zayed City,Sheikh Zayed City,Giza
2,3,Azha North,NaN,Ras Al Hekma,North Coast
3,4,Taj City,5th Settlement Compounds,"The 5th Settlement, New Cairo City",Cairo
4,5,Granville,NaN,New Capital City,Cairo
5,6,Marina 5,Marina,Al Alamein,North Coast
6,7,The Icon Residence,5th Settlement Compounds,"The 5th Settlement, New Cairo City",Cairo
7,8,La vista Ras El Hikma,NaN,Ras Al Hekma,North Coast
8,9,Azad,5th Settlement Compounds,"The 5th Settlement, New Cairo City",Cairo
9,10,Jirian,New Zayed City,Sheikh Zayed City,Giza


3.Dim_Payment

In [116]:
# Create Dim_Payment
Dim_Payment = (
    Gold_df[
        [
            "payment_method",
            "has_down_payment"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Create surrogate key
Dim_Payment.insert(0, "payment_key", range(1, len(Dim_Payment) + 1))

Dim_Payment.head()

,payment_key,payment_method,has_down_payment
0,1,Cash,True
1,2,Installments,True
2,3,Installments,False
3,4,Cash,False
4,5,Unknown,False


4.Dim_Property

In [122]:
# Create Dim_Property
Dim_Property = (
    Gold_df[
        [
            "type",
            "price_category",
            "size_category",
            "bedroom_category",
            "price_per_sqm_category"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Create surrogate key
Dim_Property.insert(0, "property_type_key", range(1, len(Dim_Property) + 1))

Dim_Property.head()

,property_type_key,type,price_category,size_category,bedroom_category,price_per_sqm_category
0,1,Chalet,Premium,Small,Small,Luxury
1,2,Villa,Luxury,Large,Family,Luxury
2,3,Chalet,Luxury,Medium,Small,Luxury
3,4,Apartment,Luxury,Large,Family,Premium
4,5,Villa,Luxury,Luxury,Luxury,Luxury


### **Fact table**

Merging keys to create relationships

In [119]:
# Start from Gold layer
Fact_Property = Gold_df.copy()

# =============================
# Add Date Key
# =============================
Fact_Property = Fact_Property.merge(
    Dim_Date,
    on=[
        "available_from",
        "available_year",
        "available_month",
        "available_month_name",
        "available_quarter",
        "available_day"
    ],
    how="left"
)

# =============================
# Add Location Key
# =============================
Fact_Property = Fact_Property.merge(
    Dim_Location,
    on=[
        "compound",
        "district",
        "city",
        "governorate"
    ],
    how="left"
)

# =============================
# Add Payment Key
# =============================
Fact_Property = Fact_Property.merge(
    Dim_Payment,
    on=[
        "payment_method",
        "has_down_payment"
    ],
    how="left"
)

# =============================
# Add Property Key
# =============================
Fact_Property = Fact_Property.merge(
    Dim_Property,
    on=[
        "type",
        "price_category",
        "size_category",
        "bedroom_category",
        "price_per_sqm_category"
    ],
    how="left"
)

# =============================
# Keep only Fact columns
# =============================
Fact_Property = Fact_Property[
    [
        # Primary Key
        "property_id",

        # Foreign Keys
        "date_key",
        "location_key",
        "payment_key",
        "property_type_key",

        # Measures
        "price",
        "down_payment",
        "down_payment_percentage",
        "price_per_sqm",

        # Numeric Property Attributes
        "size_sqm",
        "bedrooms",
        "bathrooms",

        # Boolean Attributes
        "maid_room",
        "has_down_payment",

        # Availability
        "days_until_available"
    ]
]

Fact_Property.head()

In [121]:
Fact_Property.head()

,property_id,date_key,location_key,payment_key,property_type_key,price,size_sqm,price_per_sqm,bedrooms,bathrooms,maid_room,down_payment,down_payment_percentage,has_down_payment,days_until_available,availability_status,price_per_bedroom,sqm_per_bedroom
0,1,253,1,1,1,"8,000,000.00",68,"117,647.06",1.00,1.00,True,1200000,0.15,True,0.00,Available,"8,000,000.00",68.00
1,2,255,2,1,2,"25,000,000.00",220,"113,636.36",4.00,4.00,False,2100000,0.08,True,0.00,Available,"6,250,000.00",55.00
2,3,241,3,1,3,"15,135,000.00",118,"128,262.71",2.00,2.00,False,1513000,0.10,True,0.00,Available,"7,567,500.00",59.00
3,4,248,4,2,4,"12,652,000.00",166,"76,216.87",3.00,2.00,False,1260000,0.10,True,0.00,Available,"4,217,333.33",55.33
4,5,255,5,1,5,"45,250,000.00",400,"113,125.00",7.00,7.00,False,2262500,0.05,True,0.00,Available,"6,464,285.71",57.14


#### 

In [125]:
Dim_Payment["payment_key"].is_unique

True

In [127]:
Dim_Location["location_key"].is_unique

True

In [128]:
Dim_Date["date_key"].is_unique

True

In [129]:
Fact_Property.isna().sum()

property_id                    0
date_key                       0
location_key                   0
payment_key                    0
property_type_key              0
price                          0
size_sqm                       0
price_per_sqm                  0
bedrooms                      19
bathrooms                     19
maid_room                      0
down_payment               13526
down_payment_percentage    13526
has_down_payment               0
days_until_available         514
availability_status            0
price_per_bedroom             19
sqm_per_bedroom               19
dtype: int64

In [126]:
Dim_Property["property_type_key"].is_unique

True

## **Saving all tables as csv files**

In [123]:
# Save Fact table
Fact_Property.to_csv(
    "/Users/folder/project_real_estate/data/raw/fact_property.csv",
    index=False
)

# Save Dimensions
Dim_Date.to_csv(
    "/Users/folder/project_real_estate/data/raw/dim_date.csv",
    index=False
)

Dim_Location.to_csv(
    "/Users/folder/project_real_estate/data/raw/dim_location.csv",
    index=False
)

Dim_Property.to_csv(
    "/Users/folder/project_real_estate/data/raw/dim_property.csv",
    index=False
)

Dim_Payment.to_csv(
    "/Users/folder/project_real_estate/data/raw/dim_payment.csv",
    index=False
)

## **Printing successful execution message**

In [124]:
print("Gold layer added successfully!")

Gold layer added successfully!
